<a href="https://colab.research.google.com/github/mrxy56/Information-Retrieval-Methods/blob/main/Information_Retrieval_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip -q install sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 122.2 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
import faiss

from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups

In [9]:
categories = [
    "comp.graphics",
    "comp.os.ms-windows.misc",
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.windows.x",
    "rec.autos",
    "rec.motorcycles",
    "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.space"
]

data = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

documents = []
labels = []

for label in range(10):
    ids = np.where(data.target == label)[0][:100]

    for i in ids:
        documents.append(data.data[i])
        labels.append(label)

rng = np.random.default_rng(42)
order = rng.permutation(len(documents))

documents = [documents[i] for i in order]
labels = [labels[i] for i in order]

doc_ids = [f"doc_{i}" for i in range(len(documents))]

queries = {
    "q1": "computer graphics images rendering visualization",
    "q2": "Microsoft Windows operating system software",
    "q3": "IBM PC hardware computer components",
    "q4": "Apple Macintosh computer hardware",
    "q5": "X Window graphical user interface",
    "q6": "cars automobiles engines driving",
    "q7": "motorcycles bikes riders engines",
    "q8": "baseball players teams games",
    "q9": "ice hockey players teams matches",
    "q10": "space exploration astronomy NASA spacecraft"
}

qrels = {
    f"q{i+1}": {
        doc_ids[j]
        for j, label in enumerate(labels)
        if label == i
    }
    for i in range(10)
}

doc_category = {
    doc_ids[i]: categories[labels[i]]
    for i in range(len(doc_ids))
}

print("Documents:", len(documents))
print("Queries:", len(queries))

Documents: 1000
Queries: 10


In [10]:
tokenized_documents = [
    document.lower().split()
    for document in documents
]

bm25 = BM25Okapi(tokenized_documents)

bm25_results = {}

for qid, query in queries.items():
    scores = bm25.get_scores(query.lower().split())
    ranking = np.argsort(scores)[::-1][:10]

    bm25_results[qid] = [
        doc_ids[i]
        for i in ranking
    ]

In [11]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

doc_embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

query_embeddings = model.encode(
    list(queries.values()),
    normalize_embeddings=True
).astype("float32")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [12]:
index = faiss.IndexFlatIP(
    doc_embeddings.shape[1]
)

index.add(doc_embeddings)

scores, indices = index.search(
    query_embeddings,
    10
)

dense_results = {
    qid: [
        doc_ids[i]
        for i in indices[row]
    ]
    for row, qid in enumerate(queries)
}

print("Embedding dimension:", doc_embeddings.shape[1])
print("Vectors in FAISS:", index.ntotal)

Embedding dimension: 384
Vectors in FAISS: 1000


In [13]:
def rrf(rankings, k=60):
    scores = defaultdict(float)

    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (k + rank)

    return [
        doc_id
        for doc_id, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:10]
    ]

hybrid_results = {
    qid: rrf([
        bm25_results[qid],
        dense_results[qid]
    ])
    for qid in queries
}

In [14]:
def precision_at_5(ranking, relevant):
    return sum(
        doc_id in relevant
        for doc_id in ranking[:5]
    ) / 5

def recall_at_5(ranking, relevant):
    return sum(
        doc_id in relevant
        for doc_id in ranking[:5]
    ) / len(relevant)

def mrr(ranking, relevant):
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1 / rank

    return 0

In [15]:
def evaluate(results, name):
    rows = []

    for qid in queries:
        rows.append({
            "P@5": precision_at_5(
                results[qid],
                qrels[qid]
            ),
            "Recall@5": recall_at_5(
                results[qid],
                qrels[qid]
            ),
            "MRR": mrr(
                results[qid],
                qrels[qid]
            )
        })

    averages = pd.DataFrame(rows).mean()

    return {
        "Retriever": name,
        "P@5": averages["P@5"],
        "Recall@5": averages["Recall@5"],
        "MRR": averages["MRR"]
    }

results_table = pd.DataFrame([
    evaluate(bm25_results, "BM25"),
    evaluate(dense_results, "Dense"),
    evaluate(hybrid_results, "Hybrid")
])

results_table

,Retriever,P@5,Recall@5,MRR
0,BM25,0.72,0.036,0.733333
1,Dense,0.94,0.047,1.000000
2,Hybrid,0.82,0.041,0.950000


In [16]:
comparison = []

for qid in ["q1", "q6", "q10"]:
    comparison.append({
        "Query": queries[qid],
        "BM25": bm25_results[qid][:5],
        "Dense": dense_results[qid][:5],
        "Hybrid": hybrid_results[qid][:5]
    })

pd.DataFrame(comparison)

,Query,BM25,Dense,Hybrid
0,computer graphics images rendering visualization,"[doc_312, doc_857, doc_735, doc_503, doc_850]","[doc_646, doc_947, doc_391, doc_275, doc_178]","[doc_156, doc_312, doc_646, doc_857, doc_947]"
1,cars automobiles engines driving,"[doc_751, doc_810, doc_472, doc_871, doc_10]","[doc_15, doc_285, doc_255, doc_273, doc_716]","[doc_810, doc_751, doc_15, doc_285, doc_472]"
2,space exploration astronomy NASA spacecraft,"[doc_110, doc_840, doc_584, doc_944, doc_989]","[doc_944, doc_508, doc_795, doc_989, doc_584]","[doc_944, doc_584, doc_840, doc_989, doc_795]"


In [17]:
qid = "q10"

result_table = pd.DataFrame({
    "Rank": range(1, 11),
    "Document ID": hybrid_results[qid],
    "Retrieved Category": [
        doc_category[doc_id]
        for doc_id in hybrid_results[qid]
    ],
    "Relevant": [
        doc_id in qrels[qid]
        for doc_id in hybrid_results[qid]
    ]
})

result_table

,Rank,Document ID,Retrieved Category,Relevant
0,1,doc_944,sci.space,True
1,2,doc_584,sci.space,True
2,3,doc_840,sci.space,True
3,4,doc_989,sci.space,True
4,5,doc_795,sci.space,True
5,6,doc_110,sci.space,True
6,7,doc_508,sci.space,True
7,8,doc_426,sci.space,True
8,9,doc_728,sci.space,True
9,10,doc_104,sci.space,True


In [18]:
qid = "q10"

top_docs = hybrid_results[qid][:3]

context = "\n\n".join(
    documents[int(doc_id.split("_")[1])]
    for doc_id in top_docs
)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{queries[qid]}

Answer:
"""

print(prompt)


Answer the question using only the context below.

Context:
Archive-name: space/new_probes
Last-modified: $Date: 93/04/01 14:39:17 $

UPCOMING PLANETARY PROBES - MISSIONS AND SCHEDULES

    Information on upcoming or currently active missions not mentioned below
    would be welcome. Sources: NASA fact sheets, Cassini Mission Design
    team, ISAS/NASDA launch schedules, press kits.


    ASUKA (ASTRO-D) - ISAS (Japan) X-ray astronomy satellite, launched into
    Earth orbit on 2/20/93. Equipped with large-area wide-wavelength (1-20
    Angstrom) X-ray telescope, X-ray CCD cameras, and imaging gas
    scintillation proportional counters.


    CASSINI - Saturn orbiter and Titan atmosphere probe. Cassini is a joint
    NASA/ESA project designed to accomplish an exploration of the Saturnian
    system with its Cassini Saturn Orbiter and Huygens Titan Probe. Cassini
    is scheduled for launch aboard a Titan IV/Centaur in October of 1997.
    After gravity assists of Venus, Earth and Jup

In [19]:
results_table.to_csv(
    "retrieval_results.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [20]:
import math
import numpy as np
import pandas as pd

runs = {
    "BM25": bm25_results,
    "Dense": dense_results,
    "Hybrid": hybrid_results
}

assert all(set(run) == set(queries) for run in runs.values())
assert all(set(result) <= set(doc_ids)
           for run in runs.values()
           for result in run.values())

print("Queries:", len(queries))
print("Judged query-document pairs:", sum(len(x) for x in qrels.values()))
print("Relevant documents:", sum(len(x) for x in qrels.values()))
print("Runs:", len(runs))


def precision_at_k(ranking, relevant, k):
    return sum(doc in relevant for doc in ranking[:k]) / k


def recall_at_k(ranking, relevant, k):
    return sum(doc in relevant for doc in ranking[:k]) / len(relevant)


def reciprocal_rank(ranking, relevant):
    for rank, doc in enumerate(ranking, 1):
        if doc in relevant:
            return 1 / rank
    return 0


def ndcg_at_k(ranking, relevant, k):
    labels = [
        int(doc in relevant)
        for doc in ranking[:k]
    ]

    dcg = sum(
        label / math.log2(rank + 1)
        for rank, label in enumerate(labels, 1)
    )

    ideal_count = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )

    return dcg / idcg if idcg else 0


rows = []

for system, run in runs.items():
    for qid in queries:
        ranking = run[qid]
        relevant = qrels[qid]

        rows.append({
            "Retriever": system,
            "Query ID": qid,
            "P@5": precision_at_k(ranking, relevant, 5),
            "Recall@5": recall_at_k(ranking, relevant, 5),
            "RR": reciprocal_rank(ranking, relevant),
            "nDCG@10": ndcg_at_k(ranking, relevant, 10)
        })

per_query = pd.DataFrame(rows)

aggregate = (
    per_query
    .groupby("Retriever")[["P@5", "Recall@5", "RR", "nDCG@10"]]
    .mean()
    .rename(columns={"RR": "MRR"})
    .reset_index()
)

display(aggregate.round(4))

per_query.to_csv("per_query_results.csv", index=False)
aggregate.to_csv("aggregate_results.csv", index=False)

Queries: 10
Judged query-document pairs: 1000
Relevant documents: 1000
Runs: 3


,Retriever,P@5,Recall@5,MRR,nDCG@10
0,BM25,0.72,0.036,0.7333,0.7017
1,Dense,0.94,0.047,1.0000,0.9170
2,Hybrid,0.82,0.041,0.9500,0.8285


In [21]:
qid = "q10"
system = "Hybrid"

ranking = runs[system][qid]
relevant = qrels[qid]
top5 = ranking[:5]

manual_table = pd.DataFrame({
    "Rank": range(1, 6),
    "Document ID": top5,
    "Category": [doc_category[doc] for doc in top5],
    "Relevant": [doc in relevant for doc in top5]
})

display(manual_table)

number_relevant = sum(doc in relevant for doc in top5)
first_relevant_rank = next(
    (rank for rank, doc in enumerate(ranking, 1) if doc in relevant),
    None
)

p5 = number_relevant / 5
r5 = number_relevant / len(relevant)
rr = 1 / first_relevant_rank if first_relevant_rank else 0
ndcg5 = ndcg_at_k(ranking, relevant, 5)

print(f"P@5 = {number_relevant}/5 = {p5:.3f}")
print(f"Recall@5 = {number_relevant}/{len(relevant)} = {r5:.3f}")
print(f"RR = 1/{first_relevant_rank} = {rr:.3f}")
print(f"nDCG@5 = {ndcg5:.3f}")

,Rank,Document ID,Category,Relevant
0,1,doc_944,sci.space,True
1,2,doc_584,sci.space,True
2,3,doc_840,sci.space,True
3,4,doc_989,sci.space,True
4,5,doc_795,sci.space,True


P@5 = 5/5 = 1.000
Recall@5 = 5/100 = 0.050
RR = 1/1 = 1.000
nDCG@5 = 1.000


In [22]:
scores = per_query.pivot(
    index="Query ID",
    columns="Retriever",
    values="nDCG@10"
)

selected = {}
used = set()

for system in runs:
    advantage = (
        scores[system]
        - scores.drop(columns=system).max(axis=1)
    ).sort_values(ascending=False)

    qid = next(q for q in advantage.index if q not in used)
    selected[f"{system} best"] = qid
    used.add(qid)

poor_order = scores.mean(axis=1).sort_values().index
poor_qid = next((q for q in poor_order if q not in used), poor_order[0])
selected["All systems poor"] = poor_qid

summary = []

for case, qid in selected.items():
    summary.append({
        "Case": case,
        "Query ID": qid,
        "Query": queries[qid],
        "BM25 nDCG@10": scores.loc[qid, "BM25"],
        "Dense nDCG@10": scores.loc[qid, "Dense"],
        "Hybrid nDCG@10": scores.loc[qid, "Hybrid"]
    })

    print("\n", case, "-", qid, queries[qid])

    comparison = pd.DataFrame({
        system: runs[system][qid][:5]
        for system in runs
    })

    for system in runs:
        comparison[f"{system} Relevant"] = [
            doc in qrels[qid]
            for doc in runs[system][qid][:5]
        ]

    display(comparison)

error_summary = pd.DataFrame(summary)
display(error_summary)

error_summary.to_csv("error_analysis.csv", index=False)


 BM25 best - q5 X Window graphical user interface


,BM25,Dense,Hybrid,BM25 Relevant,Dense Relevant,Hybrid Relevant
0,doc_724,doc_830,doc_830,True,True,True
1,doc_16,doc_39,doc_348,True,False,True
2,doc_348,doc_811,doc_932,True,False,True
3,doc_631,doc_960,doc_724,True,True,True
4,doc_542,doc_503,doc_16,True,True,True



 Dense best - q3 IBM PC hardware computer components


,BM25,Dense,Hybrid,BM25 Relevant,Dense Relevant,Hybrid Relevant
0,doc_485,doc_769,doc_995,False,True,False
1,doc_157,doc_379,doc_548,True,True,True
2,doc_986,doc_548,doc_843,False,True,False
3,doc_995,doc_193,doc_27,False,True,False
4,doc_843,doc_491,doc_485,False,True,False



 Hybrid best - q10 space exploration astronomy NASA spacecraft


,BM25,Dense,Hybrid,BM25 Relevant,Dense Relevant,Hybrid Relevant
0,doc_110,doc_944,doc_944,True,True,True
1,doc_840,doc_508,doc_584,True,True,True
2,doc_584,doc_795,doc_840,True,True,True
3,doc_944,doc_989,doc_989,True,True,True
4,doc_989,doc_584,doc_795,True,True,True



 All systems poor - q6 cars automobiles engines driving


,BM25,Dense,Hybrid,BM25 Relevant,Dense Relevant,Hybrid Relevant
0,doc_751,doc_15,doc_810,False,True,True
1,doc_810,doc_285,doc_751,True,True,False
2,doc_472,doc_255,doc_15,True,True,True
3,doc_871,doc_273,doc_285,True,True,True
4,doc_10,doc_716,doc_472,False,True,True


,Case,Query ID,Query,BM25 nDCG@10,Dense nDCG@10,Hybrid nDCG@10
0,BM25 best,q5,X Window graphical user interface,1.000000,0.751092,0.848238
1,Dense best,q3,IBM PC hardware computer components,0.205117,0.648932,0.423677
2,Hybrid best,q10,space exploration astronomy NASA spacecraft,1.000000,1.000000,1.000000
3,All systems poor,q6,cars automobiles engines driving,0.552746,0.933746,0.794883


In [24]:
sensitivity = []

for system, run in runs.items():
    sensitivity.append({
        "Retriever": system,
        "P@3": np.mean([
            precision_at_k(run[qid], qrels[qid], 3)
            for qid in queries
        ]),
        "P@5": np.mean([
            precision_at_k(run[qid], qrels[qid], 5)
            for qid in queries
        ]),
        "P@10": np.mean([
            precision_at_k(run[qid], qrels[qid], 10)
            for qid in queries
        ])
    })

sensitivity = pd.DataFrame(sensitivity)
display(sensitivity.round(4))
sensitivity.to_csv("cutoff_sensitivity.csv", index=False)

decision = aggregate.copy()

metrics = ["P@5", "Recall@5", "MRR", "nDCG@10"]

for metric in metrics:
    maximum = decision[metric].max()
    decision[metric + "_normalized"] = (
        decision[metric] / maximum
        if maximum > 0 else 0
    )

decision["Decision score"] = (
    0.35 * decision["P@5_normalized"]
    + 0.25 * decision["Recall@5_normalized"]
    + 0.25 * decision["MRR_normalized"]
    + 0.15 * decision["nDCG@10_normalized"]
)

decision = decision.sort_values(
    "Decision score",
    ascending=False
)

display(decision[[
    "Retriever",
    "P@5",
    "Recall@5",
    "MRR",
    "nDCG@10",
    "Decision score"
]].round(4))

print("Recommended model:", decision.iloc[0]["Retriever"])

,Retriever,P@3,P@5,P@10
0,BM25,0.7333,0.72,0.72
1,Dense,0.9333,0.94,0.90
2,Hybrid,0.8000,0.82,0.83


,Retriever,P@5,Recall@5,MRR,nDCG@10,Decision score
1,Dense,0.94,0.047,1.0000,0.9170,1.0000
2,Hybrid,0.82,0.041,0.9500,0.8285,0.8964
0,BM25,0.72,0.036,0.7333,0.7017,0.7577


Recommended model: Dense
